In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.func import functional_call, jacrev, vjp
import wandb
from accelerate.test_utils.testing import get_backend
from core.callbacks import WandBCallback
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
from core.estimators import BiasWithMSE
from core.models import NonLinearNetwork, LinearNetwork
from lightning.pytorch.callbacks import EarlyStopping
device, n_devices, _ = get_backend()

torch.set_float32_matmul_precision("highest")

## Linear Network with L1 / L2 penalty

In [2]:
# Instantiate the model
input_dim = 10
output_dim = 1
hidden_dim = 200

model = LinearNetwork(input_dim, output_dim, hidden_dim, 
                      l1_lambda=1., l1_smooth=0.01,
                      l2_lambda=0.0, 
                      lr=1e-3)

# fake data
N = 5000
X = torch.randn(N, input_dim)
betas = 5*torch.randn(input_dim) 
y = (betas @ X.T).view(-1, 1) + 0.5 * torch.randn(N, 1)  # Linear relationship with noise
# Create a DataLoader for training and testing
dataset = TensorDataset(X, y)
# train test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=3, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=3, drop_last=True)


logger = WandbLogger(project="inductive-bias", name="linear-net")
trainer = Trainer(max_epochs=200, 
                  logger=logger, 
                  callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss", patience=15, mode="min")], 
                  accelerator=device, 
                  devices=n_devices)
trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=test_dataloader)


Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_m

wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | linear    | Sequential | 2.4 K  | train
1 | loss_func | MSELoss    | 0      | train
-------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/inductive-bias/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/result.py:434: It is recommended to use `self.log('val/loss', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/loss,█▅▅▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇████
val/loss,█▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,137
train/loss,12.23692
trainer/global_step,8555
val/loss,12.31547


### Using per-sample gradients to estimate bias

In [3]:
from core.bias import RidgeBias, SmoothLassoBias, LassoBias, ElasticNet

# For this example, we'll use the same simulated data, but in batches
params = dict(model.named_parameters())
flattened_params = torch.cat([p.view(-1) for p in params.values()])
# buffers = dict(model.named_buffers())

# Define function that returns model output
def model_output(params, x):
    return functional_call(model, params, (x,))

# Create dataset and dataloader
dataset = TensorDataset(X, y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Instantiate model
# r_model = RidgeBias()
# wandb.init(project="inductive-bias", name="ridge-bias")
# r_model = SmoothLassoBias(smooth=0.1, alpha_init=0.5)
# wandb.init(project="inductive-bias", name="smooth-lasso-bias")
# r_model = LassoBias()
# wandb.init(project="inductive-bias", name="nonsmooth-lasso-bias")
r_model = ElasticNet()
wandb.init(project="inductive-bias", name="elastic-net")

wandb.watch(r_model)
optimizer = optim.Adam(r_model.parameters(), lr=1e-3)

# Training loop with batches
num_epochs = 200

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for X_batch, y_batch in dataloader:
        # Zero the gradients
        optimizer.zero_grad()
        flattened_params = flattened_params.detach().requires_grad_()

        # vector-Jacobian product, returns function (model_output) applied to primals (params, X) 
        # and a function that computes the vector-Jacobian product
        predictions, vjp_func = vjp(model_output, params, X_batch)
        # vjp with residuals, then select the derivative w.r.t. params
        # factor of 2 comes from gradient of the mse loss
        vjp_result = vjp_func(2*(y_batch - predictions))[0] # is this sign convention correct?
        true_gradients_batch = torch.cat([v.view(-1) for v in vjp_result.values()]) / batch_size

        # Compute predicted gradient ∇R(A, θ) via autograd
        R_val = r_model(flattened_params)
        gradients = torch.autograd.grad(R_val, flattened_params, create_graph=True)[0]
        # Compute loss for this batch
        loss = torch.nn.functional.mse_loss(
            gradients, 
            true_gradients_batch, 
            reduction='mean')
        
        # Backpropagate and update θ
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1

        ## Early stopping

    
    avg_epoch_loss = epoch_loss / batch_count
    # print(f"Epoch {epoch}: Average Loss = {avg_epoch_loss:.8f}")
    wandb.log({"loss": avg_epoch_loss})
    # wandb.log({"learning_rate": optimizer.param_groups[0]['lr']})
    # Learned parameters
    for name, param in r_model.named_parameters():
        # print(f"Parameter {name}: {param.detach().numpy()}")
        wandb.log({name: param.detach().numpy()})

wandb.finish()

wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


loss,█▅▄▄▃▃▂▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▁▂▂▁▂▁▂▁▂▂▂▂▂
theta_1,▆▄▃▂▂▁▁▂▂▄▄▆▆▆▇▇▇▇▇▇▇███████████████████
theta_2,█▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
loss,0.00641
theta_1,0.03643
theta_2,-4.85637


In [9]:
from core.bias import ElasticNet
from importlib import reload
from core import estimators
reload(estimators)

dataset = TensorDataset(X, y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=3)
bias_estimator = BiasWithMSE(
    predictive_model=model,
    bias_model=ElasticNet(smooth=0.01),
    grad_match_loss_fn=torch.nn.functional.mse_loss,
    lr=1e-2
)
bias_trainer = Trainer(
    max_epochs=500,
    accelerator=device,
    # devices=0,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=50, mode="min"),
    ],
    logger=WandbLogger(project="inductive-bias", name="linear-ridge"),
)
bias_trainer.fit(bias_estimator, train_dataloaders=dataloader)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name             | Type          | Params | Mode 
-----------------------------------------------------------
0 | predictive_model | LinearNetwork | 2.4 K  | train
1 | bias_model       | ElasticNet    | 2      | train
-----------------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/inductive-bias/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


bias/theta_1,▂▁▁▅▅▇▇▇▇▇▇█▇▇▇▇▇▇▇██▇█▇██▇▇█▇██▇███████
bias/theta_2,█▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇██
train/loss,▃█▅▃▄▂▃▃▅▂▅▃▄█▅▂▂▃▄▃▂▂▄▇▁▃▂▇▂▄▅▁▁▅▃▂▃▃▄▂
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
bias/theta_1,0.06576
bias/theta_2,-4.69549
epoch,92
train/loss,0.00582
trainer/global_step,14499


In [10]:
print(torch.exp(bias_estimator.bias_model.theta_1))
torch.exp(bias_estimator.bias_model.theta_2)

tensor(1.0278, grad_fn=<ExpBackward0>)


tensor(0.0089, grad_fn=<ExpBackward0>)

#### Testing vjp

In [8]:
params = dict(model.named_parameters())
# buffers = dict(model.named_buffers())

# Define function that returns model output
def model_output(params, x):
    return functional_call(model, params, (x,))

# def single_output(params, x_single):
#     # Return a scalar prediction per example 
#     return functional_call(model, params, (x_single,)).squeeze()

# Compute Jacobian of output w.r.t. weights
# x = torch.randn(10, input_dim)
gradient_wrt_params = jacrev(model_output)(params, X[:10])
# grad_per_sample = vmap(grad(model_output), in_dims=(None, 0))(params, X)
predictions = model_output(params, X[:10])
product = (-(y[:10]-predictions)).T @ gradient_wrt_params['linear.0.weight'].view(10, -1)
product.view(100, -1)

tensor([[ 2.9800e-07, -1.0095e-06,  1.1149e-06,  ...,  1.0358e-07,
         -3.1128e-07, -5.1813e-08],
        [-1.9472e-05,  6.5962e-05, -7.2847e-05,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -8.7971e-07,
          2.6438e-06,  4.4006e-07],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  2.1277e-07,
         -6.3943e-07, -1.0643e-07],
        [-1.8221e-05,  6.1724e-05, -6.8166e-05,  ..., -1.4941e-07,
          4.4902e-07,  7.4739e-08],
        [-6.1961e-06,  2.0989e-05, -2.3180e-05,  ...,  1.7927e-07,
         -5.3876e-07, -8.9677e-08]], grad_fn=<ViewBackward0>)

In [9]:
# Compute Jacobian of output w.r.t. weights
# x = torch.randn(10, input_dim)
# gradient_wrt_params = jacrev(model_output)(params, X[:10])
with torch.no_grad():
    # vector-Jacobian product, returns function (model_output) applied to primals (params, X) 
    # and a function that computes the vector-Jacobian product
    predictions, vjp_func = vjp(model_output, params, X[:10])
    # vjp with residuals, then select the derivative w.r.t. params
    vjp_result =  vjp_func(-(y[:10] - predictions))[0] 
print(vjp_result['linear.0.weight'])

tensor([[ 2.9800e-07, -1.0095e-06,  1.1149e-06,  ..., -6.9756e-08,
          2.0964e-07,  3.4894e-08],
        [-4.4249e-07,  1.4989e-06, -1.6554e-06,  ...,  1.0358e-07,
         -3.1128e-07, -5.1813e-08],
        [-1.9472e-05,  6.5962e-05, -7.2847e-05,  ...,  4.5580e-06,
         -1.3698e-05, -2.2801e-06],
        ...,
        [ 6.3829e-07, -2.1622e-06,  2.3879e-06,  ..., -1.4941e-07,
          4.4902e-07,  7.4739e-08],
        [-6.1961e-06,  2.0989e-05, -2.3180e-05,  ...,  1.4504e-06,
         -4.3588e-06, -7.2553e-07],
        [-7.6586e-07,  2.5943e-06, -2.8651e-06,  ...,  1.7927e-07,
         -5.3876e-07, -8.9677e-08]])
